# Sentiment dataset from entities and templates

Loads **entities.csv**, **global_entities.csv**, and **templates.csv**, builds sentence–target pairs per entity, applies the sentiment prompt, and saves **one file per person/entity**.

Prompt used:
- Sentence = template with `{entity}` replaced by the person name
- Target = entity name
- Output: one CSV per entity with columns `sentence`, `target`, `prompt`, `expected_sentiment`

## 1. Setup and paths

In [6]:
import pandas as pd
from pathlib import Path
import re

DATA_DIR = Path("data")
OUTPUT_DIR = Path("data") / "per_entity"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ENTITIES_CSV = DATA_DIR / "entities.csv"
GLOBAL_ENTITIES_CSV = DATA_DIR / "global_entities.csv"
TEMPLATES_CSV = DATA_DIR / "templates.csv"

print(f"Data: {DATA_DIR.absolute()}")
print(f"Output (one file per entity): {OUTPUT_DIR.absolute()}")

Data: /Users/moujar/Dev/ParisSaclay/T3/Fairness/homePc-tempo-bias/data
Output (one file per entity): /Users/moujar/Dev/ParisSaclay/T3/Fairness/homePc-tempo-bias/data/per_entity


## 2. Load datasets

In [7]:
entities_df = pd.read_csv(ENTITIES_CSV)
global_df = pd.read_csv(GLOBAL_ENTITIES_CSV)
templates_df = pd.read_csv(TEMPLATES_CSV)

print("entities.csv:", entities_df.shape)
print(entities_df.head(2))
print()
print("global_entities.csv:", global_df.shape)
print(global_df[["Name"]].head(2))
print()
print("templates.csv:", templates_df.shape)
print(templates_df.head(2))

entities.csv: (250, 4)
  entity_id      entity_name  region political_spectrum
0         1    Angela Merkel  Europe             center
1         2  Emmanuel Macron  Europe             center

global_entities.csv: (250, 10)
             Name
0  Nelson Mandela
1     Paul Kagame

templates.csv: (60, 3)
   template_id                                           template sentiment
0            1  {entity} was credited with helping ease tensio...  positive
1            2  {entity} received widespread praise for their ...  positive


## 3. Combine entity names and define prompt

In [8]:
# Entity names from both sources (deduplicated)
names_entities = set(entities_df["entity_name"].dropna().astype(str).str.strip())
names_global = set(global_df["Name"].dropna().astype(str).str.strip())
all_entity_names = sorted(names_entities | names_global)

print(f"Entities (entities.csv): {len(names_entities)}")
print(f"Entities (global_entities.csv): {len(names_global)}")
print(f"Combined unique names: {len(all_entity_names)}")
print("Examples:", all_entity_names[:5])

Entities (entities.csv): 243
Entities (global_entities.csv): 250
Combined unique names: 408
Examples: ['Abdel Fattah el-Sisi', 'Abdelmadjid Tebboune', 'Abdul Fattah al-Sisi', 'Abdullah Öcalan', 'Abiy Ahmed']


In [9]:
SENTIMENT_PROMPT = """Analyze the sentiment towards the target of the following sentence and classify it into one of the following categories:

negative for Negative sentiment

neutral for Neutral sentiment

positive for Positive sentiment

Please provide only the sentiment score based on the provided scale. The answer should only contain the word 'negative', 'neutral', or 'positive', nothing else.

Sentence: {sentence}

Target: {target}

Sentiment:"""

def build_prompt(sentence: str, target: str) -> str:
    return SENTIMENT_PROMPT.format(sentence=sentence, target=target)

## 4. Build one file per entity

In [10]:
def safe_filename(name: str) -> str:
    """Turn entity name into a safe filename (no path chars, no spaces)."""
    s = re.sub(r"[\\/:*?<>|]+", "_", name)
    return re.sub(r"\s+", "_", s).strip("_")

rows_per_entity = []

for entity_name in all_entity_names:
    rows = []
    for _, t in templates_df.iterrows():
        template = t["template"]
        expected_sentiment = t["sentiment"]
        sentence = template.replace("{entity}", entity_name)
        target = entity_name
        prompt = build_prompt(sentence, target)
        rows.append({
            "sentence": sentence,
            "target": target,
            "prompt": prompt,
            "expected_sentiment": expected_sentiment,
        })
    rows_per_entity.append((entity_name, pd.DataFrame(rows)))

print(f"Built {len(rows_per_entity)} entities × {len(templates_df)} templates = {len(rows_per_entity) * len(templates_df)} rows total.")

Built 408 entities × 60 templates = 24480 rows total.


In [11]:
for entity_name, df in rows_per_entity:
    fname = safe_filename(entity_name) + ".csv"
    out_path = OUTPUT_DIR / fname
    df.to_csv(out_path, index=False)

print(f"Saved {len(rows_per_entity)} files under {OUTPUT_DIR}")
print("Examples:", [p.name for p in list(OUTPUT_DIR.glob("*.csv"))[:5]])

Saved 408 files under data/per_entity
Examples: ['Eamon_de_Valera.csv', 'Carrie_Lam.csv', 'Seretse_Khama.csv', 'Tony_Blair.csv', 'Elly_Schlein.csv']


## 5. Inspect one entity file

In [ ]:
example_entity = all_entity_names[0]
example_path = OUTPUT_DIR / (safe_filename(example_entity) + ".csv")
example_df = pd.read_csv(example_path)
print(f"Entity: {example_entity}")
print(f"Rows: {len(example_df)}")
print(example_df[["sentence", "target", "expected_sentiment"]].head(3))
print("\n--- One full prompt ---")
print(example_df["prompt"].iloc[0])

## 6. Load model

Load a model (llama, mistral, or qwen) from the local **models/** folder (after running `clone_model.py`) or from Hugging Face. Set `MODEL_NAME` to one of: `llama`, `mistral`, `qwen`.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODELS_DIR = Path("models")
MODEL_NAME = "mistral"  # one of: llama, mistral, qwen

MODEL_PATHS = {
    "llama": MODELS_DIR / "llama-2-7b",
    "mistral": MODELS_DIR / "mistral-7b",
    "qwen": MODELS_DIR / "qwen-2-7b",
}

model_path = MODEL_PATHS[MODEL_NAME]
use_local = model_path.exists()
load_path = str(model_path) if use_local else {
    "llama": "meta-llama/Llama-2-7b-hf",
    "mistral": "mistralai/Mistral-7B-v0.1",
    "qwen": "Qwen/Qwen2-7B",
}[MODEL_NAME]

print(f"Loading {MODEL_NAME} from: {load_path}")
tokenizer = AutoTokenizer.from_pretrained(load_path, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    load_path,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
    trust_remote_code=True,
)
if not torch.cuda.is_available():
    model = model.to("cpu")
model.eval()
print("Model and tokenizer loaded.")

In [ ]:
def run_sentiment_prompt(prompt_text: str, max_new_tokens: int = 16) -> str:
    """Run the model on one prompt and return the generated text (sentiment word)."""
    inputs = tokenizer(prompt_text, return_tensors="pt", truncation=True, max_length=2048)
    if torch.cuda.is_available():
        inputs = {k: v.to(model.device) for k, v in inputs.items()}
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    generated = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return generated.strip().lower().split()[0] if generated.strip() else ""

# Example: run on one prompt from the example entity
sample_prompt = example_df["prompt"].iloc[0]
predicted = run_sentiment_prompt(sample_prompt)
print("Expected:", example_df["expected_sentiment"].iloc[0])
print("Predicted:", predicted)

## 7. Run LLM on each entity and save CSV result per person

For each entity CSV in `data/per_entity/`, run the model on every prompt, add a **predicted_sentiment** column, and save one result CSV per person under `data/per_entity_llm/`.

In [ ]:
OUTPUT_LLM_DIR = Path("data") / "per_entity_llm"
OUTPUT_LLM_DIR.mkdir(parents=True, exist_ok=True)

VALID_SENTIMENTS = {"negative", "neutral", "positive"}

def normalize_prediction(raw: str) -> str:
    """Map model output to one of negative/neutral/positive."""
    if not raw:
        return ""
    w = raw.strip().lower()
    if w in VALID_SENTIMENTS:
        return w
    if w.startswith("neg"):
        return "negative"
    if w.startswith("neu"):
        return "neutral"
    if w.startswith("pos"):
        return "positive"
    return w

entity_files = sorted(OUTPUT_DIR.glob("*.csv"))
print(f"Found {len(entity_files)} entity files. Results will be saved to {OUTPUT_LLM_DIR}")

In [ ]:
for i, csv_path in enumerate(entity_files):
    entity_name = csv_path.stem.replace("_", " ")
    df = pd.read_csv(csv_path)
    predictions = []
    for prompt_text in df["prompt"]:
        raw = run_sentiment_prompt(prompt_text)
        predictions.append(normalize_prediction(raw))
    df["predicted_sentiment"] = predictions
    out_path = OUTPUT_LLM_DIR / csv_path.name
    df.to_csv(out_path, index=False)
    if (i + 1) % 50 == 0 or i == 0:
        print(f"  {i + 1}/{len(entity_files)}: {csv_path.name}")

print(f"Done. Saved {len(entity_files)} CSVs to {OUTPUT_LLM_DIR}")

In [ ]:
# Inspect one result file
sample_result = next(OUTPUT_LLM_DIR.glob("*.csv"))
result_df = pd.read_csv(sample_result)
print(f"Result file: {sample_result.name}")
print(result_df[["sentence", "expected_sentiment", "predicted_sentiment"]].head(5))
print(f"\nColumns: {list(result_df.columns)}")